# Best Practices — Starting Point

Below is a small pipeline that — for every user — computes their **answer count**, **average answer score**, and the **region** their location maps to, then keeps only users with at least five answers. The code runs and produces a correct result, but it violates several common PySpark best practices.

Your task is to refactor this code into **three separate notebooks**, mirroring the layout a production codebase would have for the same logic (one shared library + one job entry point + one test file):

* **`Best Practices - Functions.ipynb`** — defines `location_to_region` and `compute_user_region_stats` as importable functions.
* **`Best Practices - Pipeline.ipynb`** — the main job: reads the data, includes the Functions notebook via `%run`, applies the transformations, writes the result.
* **`Best Practices - Tests.ipynb`** — unit tests for `compute_user_region_stats` using `pyspark.testing.assertDataFrameEqual` ([docs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.testing.assertDataFrameEqual.html#pyspark.testing.assertDataFrameEqual)).

The five concrete issues you should fix while doing the refactor:

1. Replace `from pyspark.sql.functions import *` with `import pyspark.sql.functions as f` — eliminates shadowing of Python built-ins like `sum` / `count`.
2. Wrap the transformations in `compute_user_region_stats(users: DataFrame, answers: DataFrame, min_answers: int = 5) -> DataFrame`. Type hints make the function contract explicit and let static checkers help you.
3. Add a test for `compute_user_region_stats` using `pyspark.testing.assertDataFrameEqual`. Build tiny in-memory DataFrames as input, assert against an expected output.
4. Rewrite the Python UDF `location_to_region` in terms of native Spark functions (`when` / `otherwise` / `rlike`) — eliminates the row-by-row JVM ↔ Python serialization a Python UDF pays for.
5. Fix the caching (currently it is not used in an optimal way).

Note: in a truly production-faithful codebase the three notebooks would be three `.py` files — `transformations.py`, a job entry-point script, and `tests/test_transformations.py` — with `pytest` running the tests in CI on every PR. The three-notebook structure here is a faithful intermediate that maps directly onto that layout.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StringType

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Best Practices - Starting Point')
    .getOrCreate()
)

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

users_input_path = os.path.join(project_path, 'data/users')
answers_input_path = os.path.join(project_path, 'data/answers')

## The pipeline to refactor

In [ ]:
@udf(returnType=StringType())
def location_to_region(location):
    if location is None:
        return 'unknown'
    location_lower = location.lower()
    europe = ['france', 'germany', 'norway', 'sweden', 'czech', 'poland', 'spain', 'italy', 'netherlands']
    asia = ['china', 'japan', 'india', 'singapore', 'thailand']
    americas = ['usa', 'united states', 'brazil', 'canada', 'mexico']
    for c in europe:
        if c in location_lower:
            return 'europe'
    for c in asia:
        if c in location_lower:
            return 'asia'
    for c in americas:
        if c in location_lower:
            return 'americas'
    return 'other'

usersDF = spark.read.parquet(users_input_path)
answersDF = spark.read.parquet(answers_input_path)

answers_cached = answersDF.cache()
answers_filtered = answers_cached.filter(col('score') > 0)

result = (
    usersDF
    .join(answers_filtered, 'user_id')
    .withColumn('region', location_to_region(col('location')))
    .groupBy('user_id', 'region')
    .agg(
        count('*').alias('answer_count'),
        avg('score').alias('avg_score'),
    )
    .filter(col('answer_count') >= 5)
)

result.orderBy(desc('avg_score')).show(10)
result.groupBy('region').count().show()
result.write.mode('overwrite').format('noop').save()

In [ ]:
spark.stop()